In [ ]:
%pip install tqdm SALib plotly

# FAST-UAV - Supply Chain Integrated Uncertainty Quantification (UQ)
*Coupled Uncertainty Analysis of Drone Sizing and Supply Chain Metrics using Sobol Method*

이 노트북은 물리적 드론 모델(FAST-OAD)과 공급망 모델(Supply Chain Model)을 통합하여 불확실성 정량화(UQ)를 수행합니다.
Sobol 분석을 통해 설계 변수(예: Payload) 및 공급망 불확실성(예: 리드타임 변동)이 최종 성능(MTOW, Cost, Lead Time)에 미치는 영향을 전역 민감도 지수(Total Sensitivity Index)로 분석합니다.


In [ ]:
# Import Required Libraries
import os
import os.path as pth
import pandas as pd
import numpy as np
import shutil
import xml.etree.ElementTree as ET
import fastoad.api as oad
from SALib.sample import saltelli
from SALib.analyze import sobol
import plotly.graph_objects as go
import plotly.express as px

# Supply Chain Model Import
try:
    from fastuav.models.supply_chain.model import run_supply_chain_scenario
except ImportError:
    import sys
    current_dir = os.getcwd()
    src_path = pth.abspath(pth.join(current_dir, '..', '..'))
    if src_path not in sys.path:
        sys.path.append(src_path)
    from fastuav.models.supply_chain.model import run_supply_chain_scenario

# Visualization helper
import matplotlib.pyplot as plt

# Increase display width
from IPython.display import display, HTML
display(HTML("<style>.container { width:95% !important; }</style>"))

In [ ]:
# --- Path Setup ---
DATA_FOLDER_PATH = "./data"
WORK_FOLDER_PATH = "./workdir_uq" # Separate workdir for UQ
CONFIGURATION_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "configurations")
SOURCE_FOLDER_PATH = pth.join(DATA_FOLDER_PATH, "source_files")

if not os.path.exists(WORK_FOLDER_PATH):
    os.makedirs(WORK_FOLDER_PATH)

# Configuration Files
CONFIGURATION_FILE = pth.join(CONFIGURATION_FOLDER_PATH, "multirotor_mdo.yaml")
SOURCE_FILE = pth.join(SOURCE_FOLDER_PATH, "problem_inputs_quadcopter.xml")

# Supply Chain Input Files
catalog_csv = pth.join(SOURCE_FOLDER_PATH, 'supplier_parts_catalog_sample.csv')
map_csv = pth.join(SOURCE_FOLDER_PATH, 'bom_to_part_family_map_sample.csv')
assump_csv = pth.join(SOURCE_FOLDER_PATH, 'cost_leadtime_assumptions_sample.csv')

# Load Baseline SC Data
catalog_df_base = pd.read_csv(catalog_csv)
map_df_base = pd.read_csv(map_csv)
assump_df_base = pd.read_csv(assump_csv)

print("Setup Complete.")
print(f"Working Directory: {WORK_FOLDER_PATH}")

In [ ]:
def generate_bom_from_output(output_xml_path):
    """
    Generate DataFrame BOM from FAST-OAD output XML.
    Extracts key sizing metrics to map to supply chain items.
    """
    if not os.path.exists(output_xml_path):
        # In UQ, if optimization fails, we might not have a file
        return None
        
    root = ET.parse(output_xml_path).getroot()

    def get_float_any(paths, default=0.0):
        for p in paths:
            node = root.find(p)
            if node is not None:
                try: return float(node.text)
                except: pass
        return default

    # 1. Quantities (Assume Quadcopter for this UQ)
    n_prop = 4.0
    n_arm = 4.0

    # 2. Masses (kg)
    m_battery = get_float_any(['./data/weight/propulsion/multirotor/battery/mass'])
    m_esc = get_float_any(['./data/weight/propulsion/multirotor/esc/mass'])
    m_motor = get_float_any(['./data/weight/propulsion/multirotor/motor/mass'])
    m_prop = get_float_any(['./data/weight/propulsion/multirotor/propeller/mass'])
    
    # 3. Perf References
    battery_energy_kj = get_float_any(['./data/propulsion/multirotor/battery/energy', './data/propulsion/multirotor/battery/energy/estimated'])
    esc_max_power_w = get_float_any(['./data/propulsion/multirotor/esc/max', './data/propulsion/multirotor/esc/power/max'])
    motor_max_torque_nm = get_float_any(['./data/propulsion/multirotor/motor/torque/max', './data/propulsion/multirotor/motor/torque/max/estimated'])
    prop_diameter_m = get_float_any(['./data/propulsion/multirotor/propeller/diameter', './data/propulsion/multirotor/propeller/diameter/estimated'])
    arm_length_m = get_float_any(['./data/geometry/multirotor/arms/length', './data/geometry/arms/length'])

    rows = [
        {
            'component_id': 'battery_pack', 'part_family': 'battery', 'quantity': 1,
            'unit_mass_kg': m_battery, 'total_mass_kg': m_battery,
            'required_perf_key': 'energy_kJ', 'required_perf_min': battery_energy_kj,
        },
        {
            'component_id': 'motor', 'part_family': 'motor', 'quantity': int(n_prop),
            'unit_mass_kg': m_motor, 'total_mass_kg': m_motor * n_prop,
            'required_perf_key': 'max_torque_Nm', 'required_perf_min': motor_max_torque_nm,
        },
        {
            'component_id': 'esc', 'part_family': 'esc', 'quantity': int(n_prop),
            'unit_mass_kg': m_esc, 'total_mass_kg': m_esc * n_prop,
            'required_perf_key': 'max_power_W', 'required_perf_min': esc_max_power_w,
        },
        {
            'component_id': 'propeller', 'part_family': 'propeller', 'quantity': int(n_prop),
            'unit_mass_kg': m_prop, 'total_mass_kg': m_prop * n_prop,
            'required_perf_key': 'diameter_m', 'required_perf_min': prop_diameter_m,
        },
        {
            'component_id': 'frame_arms', 'part_family': 'frame_arms', 'quantity': int(n_arm),
            'unit_mass_kg': 0.0, 'total_mass_kg': 0.0, 
            'required_perf_key': 'arm_length_m', 'required_perf_min': arm_length_m,
        },
    ]
    return pd.DataFrame(rows)

In [ ]:
# --- 2. Define the Problem for SALib ---
# We want to vary physical params and supply chain params together.

# Problem Definition
problem = {
    'num_vars': 4,
    'names': [
        'payload_mass_kg', 
        'hover_duration_min', 
        'sc_risk_tolerance', 
        'sc_logistics_buffer'
    ],
    'bounds': [
        [0.5, 3.0],   # Payload (kg)
        [10.0, 30.0], # Hover (min)
        [1.0, 1.5],   # Risk Multiplier (1.0 = Neutral, 1.5 = High Aversion)
        [0, 10]       # Logisitics Buffer (days)
    ]
}

# Generate Samples (Saltelli)
# N should be power of 2 (e.g., 64, 128, 256)
# Total runs = N * (2D + 2) = N * (2*4 + 2) = 10N
N_SAMPLES = 64
X_samples = saltelli.sample(problem, N_SAMPLES, calc_second_order=True)

print(f"Total samples to evaluate: {len(X_samples)}")
print(f"First 5 samples:\n{X_samples[:5]}")

In [1]:
# --- 3. Define the Evaluator Function ---

def evaluate_point_for_uq(params):
    """
    Executes one simulation point.
    params = [payload, hover, risk_mult, buffer]
    """
    p_mass, h_dur, r_mult, l_buff = params
    
    # --- A. Setup FAST-OAD Input ---
    
    # 1. Reset from configuration to ensure clean state
    # We want to avoid reading/writing to the same file in parallel if we were doing parallel exec.
    # For now, serial execution is safer.
    
    input_file = pth.join(WORK_FOLDER_PATH, "problem_inputs.xml")
    if not os.path.exists(input_file):
        oad.generate_inputs(CONFIGURATION_FILE, SOURCE_FILE, overwrite=True)
        # Move it to our clean workdir if needed, but let's assume standard behavior writes to ./workdir usually
        # We need to ensure we are working in the right folder or pointing to specific files.
        # fastoad usually writes to where the command is run or specific paths.
        
    # Let's do explicit XML parsing/writing to a temporary unique file per run if we wanted parallel.
    # For serial, just overwrite problem_inputs.xml in WORK_FOLDER_PATH
    
    # Start fresh from source
    tree = ET.parse(SOURCE_FILE)
    root = tree.getroot()
    
    def set_val(name_suffix, val):
        for var in root.findall(".//variable"):
            name_node = var.find("name")
            if name_node is not None and name_node.text and name_node.text.endswith(name_suffix):
                val_node = var.find("value")
                if val_node is not None:
                    val_node.text = str(val)
                return
                
    set_val("mission:operational:main_route:payload:mass", p_mass)
    set_val("mission:sizing:main_route:hover:duration", h_dur)
    
    # Force output path to be within our UQ work folder to avoid collisions
    # But usually output path is defined in configuration YAML.
    # We might need to copy YAML and modify it? 
    # For simplicity, let's just run in serial and use standard output names.
    
    temp_input_path = pth.join(WORK_FOLDER_PATH, "current_input.xml")
    tree.write(temp_input_path)
    
    # --- B. Run Sizing Optimization ---
    try:
        # We need to tell optimize_problem to use our specific input file.
        # But optimize_problem takes a config file.
        # We can use the lower level API or just modify the expected input file.
        # Let's assume standard file locations for now.
        
        # Override the standard input file location in yaml? Too complex.
        # Just overwrite the file pointed to by the yaml?
        # The yaml usually points to relative paths.
        
        # Simple hack: Reuse the logic from previous notebook
        # Just use oad.generate_inputs to create the file structure, then overwrite values.
        
        # Assuming WORK_FOLDER_PATH/problem_inputs.xml is what the yaml reads.
        default_input_loc = pth.join(os.path.dirname(CONFIGURATION_FILE), "../../workdir/problem_inputs.xml")
        default_input_loc = pth.abspath(default_input_loc)
        
        # Write our modified tree to the location fastoad expects
        tree.write(default_input_loc)
        
        # Run
        problem = oad.optimize_problem(CONFIGURATION_FILE, overwrite=True)
        
        # Read Output
        default_output_loc = pth.join(os.path.dirname(CONFIGURATION_FILE), "../../workdir/problem_outputs.xml")
        
        if not os.path.exists(default_output_loc):
            return [np.nan, np.nan, np.nan] # Failed
            
        # Get MTOW
        out_root = ET.parse(default_output_loc).getroot()
        mtow = float(out_root.find(".//variable[name='data:weight:mtow']/value").text)
        
        # --- C. Run Supply Chain ---
        bom_df = generate_bom_from_output(default_output_loc)
        
        if bom_df is None or bom_df.empty:
            return [mtow, np.nan, np.nan]
            
        selection, summary, _ = run_supply_chain_scenario(
            bom_df=bom_df,
            catalog_df=catalog_df_base,
            map_df=map_df_base,
            risk_cost_multiplier=r_mult,
            logistics_buffer_days=int(l_buff),
            selection_strategy='min_cost',
            use_continuous_model=True
        )
        
        cost = summary[summary['metric']=='total_cost_risk_adjusted_usd']['value'].values[0]
        lead_time = summary[summary['metric']=='total_lead_time_days']['value'].values[0]
        
        return [mtow, cost, lead_time]
        
    except Exception as e:
        print(f"Error in run {params}: {e}")
        return [np.nan, np.nan, np.nan]

print("Evaluator Ready.")

Evaluator Ready.


In [22]:
# --- 4. Run Sobol Analysis (Serial Execution) ---
import time
from tqdm import tqdm

print("Running UQ Loop...")
start_time = time.time()

Y_results = []
for i, x in enumerate(tqdm(X_samples)):
    res = evaluate_point_for_uq(x)
    Y_results.append(res)
    
Y_array = np.array(Y_results)

# Create DataFrame for Analysis
df_uq = pd.DataFrame(X_samples, columns=problem['names'])
df_uq['mtow'] = Y_array[:, 0]
df_uq['cost'] = Y_array[:, 1]
df_uq['lead_time'] = Y_array[:, 2]

# Filter NaNs
df_success = df_uq.dropna()
success_rate = len(df_success) / len(df_uq)
print(f"Success Rate: {success_rate:.1%} ({len(df_success)} / {len(df_uq)})")
print(f"Elapsed Time: {time.time() - start_time:.1f}s")

df_success.to_csv(pth.join(WORK_FOLDER_PATH, "uq_results_raw.csv"), index=False)
df_success.head()

KeyboardInterrupt: 

In [ ]:
# --- 5. Sobol Indices & Visualization ---

def plot_sobol_indices(output_col_name, title):
    """Calculates and plots Sobol indices for a given output metric."""
    Y_vector = df_success[output_col_name].values
    
    # SALib Sobol Analysis
    # Need to match input samples. We filtered NaNs, so we need corresponding Xs.
    X_success = df_success[problem['names']].values
    
    # Note: Sobol requires structured sampling. If we drop random rows (from failures), 
    # the Sobol assumption might be violated slightly for strict mathematical purity, 
    # but for practical engineering it's often acceptable if failures are few and random.
    # If failures are systematic (e.g. high payload always fails), that's a different issue.
    
    # For now, let's proceed. 
    # If using Saltelli, removing samples breaks the sequence structure.
    # Ideally we should fix failures or use a different sampling method (e.g. random + regression).
    # But let's try computing indices anyway, SALib might complain or give approximate results.
    
    # Actually, SALib strictly requires N*(2D+2) rows in exact order for the specific Sobol estimator.
    # If rows are missing, we cannot use `sobol.analyze`.
    # Alternative: Use simple correlation or regression based sensitivity if many failures occur.
    
    # Let's check if we have full data.
    if len(df_success) == len(X_samples):
        Si = sobol.analyze(problem, Y_vector, calc_second_order=True)
        
        # Plot
        df_si = pd.DataFrame({
            'Parameter': problem['names'],
            'ST (Total)': Si['ST'],
            'S1 (First)': Si['S1']
        })
        
        fig = go.Figure(data=[
            go.Bar(name='Total Effect (ST)', x=df_si['Parameter'], y=df_si['ST (Total)']),
            go.Bar(name='First Order (S1)', x=df_si['Parameter'], y=df_si['S1 (First)'])
        ])
        fig.update_layout(title=f'Sobol Sensitivity Indices for {title}', 
                          yaxis_title='Sensitivity Index', barmode='group')
        fig.show()
        
        return Si
    else:
        print(f"Warning: {len(X_samples) - len(df_success)} failed runs detected.")
        print("Standard Sobol analysis requires all samples. Switching to Correlation Analysis.")
        
        # Simple Correlation
        corr = df_success.corr(method='spearman')[output_col_name].drop(output_col_name)
        # Drop other outputs if present
        for col in ['mtow', 'cost', 'lead_time']:
            if col in corr: corr = corr.drop(col)
            
        fig = px.bar(x=corr.index, y=corr.values, title=f"Spearman Correlation for {title}")
        fig.show()
        return None

# Analyze Each Metric
si_mtow = plot_sobol_indices('mtow', 'MTOW (Mass)')
si_cost = plot_sobol_indices('cost', 'Risk Adjusted Cost')
si_lead = plot_sobol_indices('lead_time', 'Lead Time')

In [ ]:
# FAST-UAV - Supply Chain Integrated Uncertainty Quantification (UQ)
*Coupled Uncertainty Analysis of Drone Sizing and Supply Chain Metrics using Sobol Method*

이 노트북은 물리적 드론 모델(FAST-OAD)과 공급망 모델(Supply Chain Model)을 통합하여 불확실성 정량화(UQ)를 수행합니다.
Sobol 분석을 통해 설계 변수 및 공급망 변수(불확실성)가 최종 성능(MTOW, Cost, Lead Time)에 미치는 영향을 전역 민감도 지수(Total Sensitivity Index)로 분석합니다.

## 1. 개요
- **목표**: 드론 설계 변수 + 공급망 불확실성 --> 비용/납기/리스크 변동성 분석
- **방법**: Saltelli Sampling + Sobol Sensitivity Analysis
- **통합**: `fastoad` (Sizing) + `fastuav.models.supply_chain` (Business)

---

In [ ]:
# --- 6. Advanced Visualization: Parallel Coordinates ---

fig_pc = px.parallel_coordinates(
    df_success, 
    color="cost", 
    labels={
        "payload_mass_kg": "Payload (kg)",
        "hover_duration_min": "Hover (min)",
        "sc_risk_tolerance": "Risk Tol.",
        "sc_logistics_buffer": "Buffer (days)",
        "mtow": "MTOW (kg)",
        "cost": "Cost ($)",
        "lead_time": "Lead Time (days)",
    },
    title="Design Parameter Impact on Cost & Performance"
)
fig_pc.show()

# Scatter Matrix
fig_sc = px.scatter_matrix(
    df_success,
    dimensions=["payload_mass_kg", "hover_duration_min", "mtow", "cost"],
    color="lead_time",
    title="Scatter Matrix showing Trade-offs"
)
fig_sc.show()